# Real Kaggle fraud detection walkthrough
Use the project virtual environment as the Jupyter kernel. No synthetic fallback. Run `fraudlab download` first.
See README for the controlled-MCAR versus clean-data distinction.

In [ ]:
from pathlib import Path
import os
if not Path('fraudlab').exists() and Path('../fraudlab').exists():
    os.chdir('..')
import json
import pandas as pd
from fraudlab.data import load, chronological_parts, inject_missing, INPUTS
from fraudlab.features import stream_features


In [ ]:
data, audit = load('data/creditcard.csv')
audit

In [ ]:
data['Class'].value_counts().rename({0: 'Legitimate', 1: 'Fraud'})

## Split before learned preprocessing or resampling
Keep the original class distribution in validation and test. No customer IDs are available, so historical features describe the global stream.

In [ ]:
parts = chronological_parts(data)
masked, manifests = inject_missing(parts, count=45000, seed=42)
pd.DataFrame([{'split': name, 'rows': len(p), 'frauds': int(p.Class.sum()), 'masked_cells': len(m)}
              for name, p, m in zip(['train','validation','test'], masked, manifests)])

In [ ]:
train_features, history = stream_features(masked[0][INPUTS])
validation_features, history = stream_features(masked[1][INPUTS], history)
train_features.head()

## Execute the complete experiment
Set RUN_TRAINING to True explicitly. Use a new output directory. Default config has a larger search than verification config.

In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    import yaml
    from fraudlab.run import run
    config = yaml.safe_load(Path('configs/verification.yaml').read_text())
    result = run('data/creditcard.csv', 'runs/notebook', config)

## Read measured results
No target metric is hard-coded. Compare the ensemble with the named baseline, and inspect warnings, confusion counts, source hash, and split manifests.

In [ ]:
report_path = Path('runs/notebook/metrics.json')
if report_path.exists():
    results = json.loads(report_path.read_text())
    display(pd.DataFrame({'ensemble': results['test'], 'RF baseline': results['baseline_test']}).drop(index='confusion_matrix'))
else:
    print('Train first, or change report_path to an existing run directory.')